In [0]:
-- WHAT IS THE SILVER LAYER?
--   ──────────────────────────
--   Silver is the cleaning room. We take the raw Bronze data and:
--     1. Convert columns to the right data types (text to number)
--     2. Remove bad values (-999 codes, nulls, impossible values)
--     3. Add the H3 geospatial index (explained below)
--     4. Extract year, month and season from dates
--
--   WHAT IS H3 GEOSPATIAL INDEXING?
--   ─────────────────────────────────
--   H3 divides the entire Earth into hexagonal grid cells.
--   Every location gets a unique hex cell ID number.
--   This lets us join datasets that do not have exactly the same
--   coordinates — as long as both points fall inside the same hex cell
--   they can be joined together.
--
--   We use three resolutions (three different hex sizes):
--     Resolution 4 = roughly 1,770 km per cell
--                    Used for joining SST to CC (they have different grids)
--     Resolution 5 = roughly 252 km per cell
--                    Used for joining animals to environment
--     Resolution 6 = roughly 36 km per cell
--                    Stored for future detailed analysis
--
--   IMPORTANT: h3_longlatash3() takes LONGITUDE first then LATITUDE
--   This is the opposite of normal geographic convention.
--   Always double check this when writing H3 queries.
--
--   COLUMN NAME NOTES (from Bronze investigation)
--   ───────────────────────────────────────────────
--   Microplastics Bronze columns were renamed during ingestion:
--     latitude_degree       (from `Latitude (degree)`)
--     longitude_degree      (from `Longitude (degree)`)
--     sample_date           (from `Sample Date`)
--     microplastics_measurement (from `Microplastics Measurement`)
--     concentration_class   (from `Concentration Class`)
--     unit                  (from `Unit`)
--
--   ENSO Bronze is in wide format (one row per year, 12 month columns).
--   Silver unpivots this to long format (one row per year/month).
----

In [0]:
--SST Silver
-- What bad values do we remove?
--   -999  = NOAA missing data code (same as writing N/A in a spreadsheet)
--   below -2 degrees C = impossible ocean temperature
--   above 40 degrees C = beyond any recorded ocean surface temperature
--   NULL = rows with no coordinates cannot be mapped or joined
--
-- How do we extract the month from the filename?
-- Each row came from a monthly file e.g. SST-Jan2009.csv
-- We use CASE to check the filename and assign the correct month number
-- Jan = 1, Feb = 2, Mar = 3 and so on

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.sst_silver AS
SELECT
CAST(latitude_raw AS DOUBLE) AS latitude,
CAST(longitude_raw AS DOUBLE) AS longitude,
CAST(sst_raw AS DOUBLE) AS sea_surface_temperature,
CAST(source_year AS INTEGER) AS year,
CASE
WHEN source_table LIKE '%Jan%' THEN 1
WHEN source_table LIKE '%Feb%' THEN 2
WHEN source_table LIKE '%Mar%' THEN 3
WHEN source_table LIKE '%Apr%' THEN 4
WHEN source_table LIKE '%May%' THEN 5
WHEN source_table LIKE '%Jun%' THEN 6
WHEN source_table LIKE '%Jul%' THEN 7
WHEN source_table LIKE '%Aug%' THEN 8
WHEN source_table LIKE '%Sep%' THEN 9
WHEN source_table LIKE '%Oct%' THEN 10
WHEN source_table LIKE '%Nov%' THEN 11
WHEN source_table LIKE '%Dec%' THEN 12
END AS month,
22 AS lat_upper_bound,
-23 AS lat_lower_bound,
126 AS long_upper_bound,
-73 AS long_lower_bound,
h3_longlatash3(CAST(longitude_raw AS DOUBLE), CAST(latitude_raw AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(longitude_raw AS DOUBLE), CAST(latitude_raw AS DOUBLE), 5) AS h3_res5,
h3_longlatash3(CAST(longitude_raw AS DOUBLE), CAST(latitude_raw AS DOUBLE), 6) AS h3_res6,
source_table,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE latitude_raw IS NOT NULL
AND longitude_raw IS NOT NULL
AND sst_raw IS NOT NULL
AND CAST(sst_raw AS DOUBLE) != -999
AND CAST(sst_raw AS DOUBLE) BETWEEN -2 AND 40
AND CAST(latitude_raw AS DOUBLE) BETWEEN -90 AND 90
AND CAST(longitude_raw AS DOUBLE) BETWEEN -180 AND 180;

In [0]:
-- VALIDATION — null_sst, null_h3 should be zero
-- min_sst should be around 15C, max_sst below 35C

SELECT
count(*) AS total_rows,
count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
round(min(sea_surface_temperature), 2) AS min_sst,
round(max(sea_surface_temperature), 2) AS max_sst,
count(DISTINCT year) AS years_loaded
FROM mcphersonsharyn_silver.sst_silver;

In [0]:
-- CC Silver
-- Same approach as SST with different value rules:
--   CC must be greater than 0 (zero means no measurement)
--   CC must be less than 100 (above 100 mg/m3 is anomalous)
--   May 2024 will have zero rows — this is expected and documented

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.cc_silver AS
SELECT
CAST(latitude_raw AS DOUBLE) AS latitude,
CAST(longitude_raw AS DOUBLE) AS longitude,
CAST(cc_raw AS DOUBLE) AS chlorophyll_concentration,
CAST(source_year AS INTEGER) AS year,
CASE
WHEN source_table LIKE '%Jan%' THEN 1
WHEN source_table LIKE '%Feb%' THEN 2
WHEN source_table LIKE '%Mar%' THEN 3
WHEN source_table LIKE '%Apr%' THEN 4
WHEN source_table LIKE '%May%' THEN 5
WHEN source_table LIKE '%Jun%' THEN 6
WHEN source_table LIKE '%Jul%' THEN 7
WHEN source_table LIKE '%Aug%' THEN 8
WHEN source_table LIKE '%Sep%' THEN 9
WHEN source_table LIKE '%Oct%' THEN 10
WHEN source_table LIKE '%Nov%' THEN 11
WHEN source_table LIKE '%Dec%' THEN 12
END AS month,
22 AS lat_upper_bound,
-23 AS lat_lower_bound,
126 AS long_upper_bound,
-73 AS long_lower_bound,
h3_longlatash3(CAST(longitude_raw AS DOUBLE), CAST(latitude_raw AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(longitude_raw AS DOUBLE), CAST(latitude_raw AS DOUBLE), 5) AS h3_res5,
source_table,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_cc_master
WHERE latitude_raw IS NOT NULL
AND longitude_raw IS NOT NULL
AND cc_raw IS NOT NULL
AND CAST(cc_raw AS DOUBLE) != -999
AND CAST(cc_raw AS DOUBLE) > 0
AND CAST(cc_raw AS DOUBLE) < 100
AND CAST(latitude_raw AS DOUBLE) BETWEEN -90 AND 90
AND CAST(longitude_raw AS DOUBLE) BETWEEN -180 AND 180;

In [0]:
-- VALIDATION — check May 2024 shows 0 rows (known source gap)

SELECT year, month, count(*) AS rows,
CASE WHEN count(*) = 0 THEN 'SOURCE GAP — no NOAA data' ELSE 'OK' END AS status
FROM mcphersonsharyn_silver.cc_silver
WHERE year = 2024
GROUP BY year, month
ORDER BY month;

In [0]:
-- ENSO Silver
-- The ENSO Bronze table is in WIDE FORMAT:
--   One row per year with 12 monthly values as separate columns
--   (jan, feb, mar, apr, may, jun, jul, aug, sep, oct, nov, dec)
--
-- Silver converts this to LONG FORMAT:
--   One row per year/month combination
--   This is done using UNION ALL — one SELECT per month
--
-- We also classify each month as El Nino, La Nina or Neutral:
--   MEI value above +0.5  = El Nino  (warmer than normal Pacific)
--   MEI value below -0.5  = La Nina  (cooler than normal Pacific)
--   Between -0.5 and +0.5 = Neutral  (normal conditions)
--
-- ENSO joins on year and month only — no location needed because
-- ENSO is a global index. Same value applies everywhere that month.
--
-- -999 values mean future months not yet recorded — excluded below

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.enso_silver AS
SELECT
year,
month,
mei_value,
CASE
WHEN mei_value > 0.5 THEN 'El Nino'
WHEN mei_value < -0.5 THEN 'La Nina'
ELSE 'Neutral'
END AS enso_phase,
current_timestamp() AS processed_timestamp
FROM (
-- Unpivot: one SELECT per month combining all years
-- Each block takes the monthly column and assigns the month number
SELECT year, 1 AS month, jan AS mei_value FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE jan IS NOT NULL AND jan != -999
UNION ALL
SELECT year, 2, feb FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE feb IS NOT NULL AND feb != -999
UNION ALL
SELECT year, 3, mar FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE mar IS NOT NULL AND mar != -999
UNION ALL
SELECT year, 4, apr FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE apr IS NOT NULL AND apr != -999
UNION ALL
SELECT year, 5, may FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE may IS NOT NULL AND may != -999
UNION ALL
SELECT year, 6, jun FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE jun IS NOT NULL AND jun != -999
UNION ALL
SELECT year, 7, jul FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE jul IS NOT NULL AND jul != -999
UNION ALL
SELECT year, 8, aug FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE aug IS NOT NULL AND aug != -999
UNION ALL
SELECT year, 9, sep FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE sep IS NOT NULL AND sep != -999
UNION ALL
SELECT year, 10, oct FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE oct IS NOT NULL AND oct != -999
UNION ALL
SELECT year, 11, nov FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE nov IS NOT NULL AND nov != -999
UNION ALL
SELECT year, 12, dec FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE dec IS NOT NULL AND dec != -999
)
WHERE year IS NOT NULL
AND mei_value IS NOT NULL;

In [0]:
-- VALIDATION — check phase distribution looks sensible
-- Should show roughly equal spread of El Nino, La Nina and Neutral
SELECT enso_phase, count(*) AS months
FROM mcphersonsharyn_silver.enso_silver
GROUP BY enso_phase;

In [0]:
-- Check our study years all have 12 months of ENSO data
SELECT year, count(*) AS months
FROM mcphersonsharyn_silver.enso_silver
WHERE year IN (2009, 2010, 2012, 2013, 2014, 2015, 2021, 2022, 2023, 2024, 2025)
GROUP BY year
ORDER BY year;

In [0]:
-- Microplastics Silver
-- Microplastics data is sparse — point observations from research cruises.
-- Most H3 cells will have NULL microplastics values in the Gold join.
-- That is normal and expected — not a pipeline error.
--
-- Column names used here match what was confirmed in Bronze:
--   latitude_degree       — renamed from `Latitude (degree)`
--   longitude_degree      — renamed from `Longitude (degree)`
--   sample_date           — renamed from `Sample Date`
--   microplastics_measurement — renamed from `Microplastics Measurement`
--   concentration_class   — renamed from `Concentration Class`
--   unit                  — renamed from `Unit`
--
-- Quality filters confirmed working in Bronze diagnostics:
--   unit = 'pieces/m3'    — excludes 1 row with non-standard units
--   coordinates within bounding box
--   concentration >= 0
--   sample_date IS NOT NULL
--
-- Note: sample_date_parsed used as output alias to avoid column name
-- conflict with the input column sample_date in the same SELECT

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.microplastics_silver AS
SELECT
CAST(OBJECTID AS BIGINT) AS record_id,
CAST(latitude_degree AS DOUBLE) AS latitude,
CAST(longitude_degree AS DOUBLE) AS longitude,
TO_DATE(sample_date, 'M/d/yyyy h:mm:ss a') AS sample_date_parsed,
YEAR(TO_DATE(sample_date, 'M/d/yyyy h:mm:ss a')) AS year,
MONTH(TO_DATE(sample_date, 'M/d/yyyy h:mm:ss a')) AS month,
ocean AS ocean,
marine_setting AS marine_setting,
sampling_method AS sampling_method,
CAST(microplastics_measurement AS DOUBLE) AS concentration,
unit AS concentration_units,
concentration_class AS pollution_severity,
short_reference AS reference,
h3_longlatash3(CAST(longitude_degree AS DOUBLE), CAST(latitude_degree AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(longitude_degree AS DOUBLE), CAST(latitude_degree AS DOUBLE), 5) AS h3_res5,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
WHERE unit = 'pieces/m3'
AND CAST(latitude_degree AS DOUBLE) BETWEEN -23 AND 22
AND CAST(longitude_degree AS DOUBLE) BETWEEN -73 AND 126
AND CAST(microplastics_measurement AS DOUBLE) >= 0
AND sample_date IS NOT NULL;
 
-- VALIDATION — expected: 470 rows
SELECT
count(*) AS total_rows,
count(DISTINCT year) AS years_covered,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
round(min(concentration), 4) AS min_concentration,
round(max(concentration), 4) AS max_concentration
FROM mcphersonsharyn_silver.microplastics_silver;
 

In [0]:
-- MICROPLASTICS AGGREGATED BY H3 CELL AND MONTH 
-- Why do we aggregate?
-- The raw microplastics table has individual point observations.
-- To join to our gridded SST and CC data we need one value per
-- H3 cell per month. We calculate the average concentration across
-- all observations in the same cell and month.
--
-- We join on month only (not year) because microplastics data is sparse
-- and we want seasonal patterns rather than exact year matches.

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.microplastics_h3_monthly AS
SELECT
h3_res4,
h3_res5,
month,
round(avg(concentration), 6) AS avg_concentration,
round(max(concentration), 6) AS max_concentration,
count(*) AS observation_count,
CASE
WHEN avg(concentration) < 0.1 THEN 'Very Low'
WHEN avg(concentration) < 1.0 THEN 'Low'
WHEN avg(concentration) < 10.0 THEN 'High'
ELSE 'Very High'
END AS pollution_severity,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_silver.microplastics_silver
WHERE concentration IS NOT NULL
GROUP BY h3_res4, h3_res5, month;
 
-- VALIDATION
SELECT count(*) AS total_rows, count(DISTINCT h3_res4) AS unique_cells
FROM mcphersonsharyn_silver.microplastics_h3_monthly;

In [0]:
-- UNIFIED ENVIRONMENTAL CONDITIONS TABLE              
-- Joins SST + CC + ENSO + Microplastics into one table
-- This is the most important table in Silver.
-- All four environmental sources come together here.
--
-- Why LEFT JOIN?
-- A LEFT JOIN(preference) keeps ALL SST rows even where no matching CC or
-- microplastics data exists. Missing values show as NULL.
-- We never lose an SST reading just because CC is missing.
--
-- Join keys:
--   SST to CC:            h3_res4 + year + month
--   SST to ENSO:          year + month only (global index, no location)
--   SST to Microplastics: h3_res4 + month only (sparse, no year match)

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.environmental_conditions_silver AS
SELECT
sst.latitude,
sst.longitude,
sst.h3_res4,
sst.h3_res5,
sst.h3_res6,
sst.sea_surface_temperature,
sst.year,
sst.month,
sst.lat_upper_bound,
sst.lat_lower_bound,
sst.long_upper_bound,
sst.long_lower_bound,
cc.chlorophyll_concentration,
enso.mei_value,
enso.enso_phase,
mp.avg_concentration AS microplastic_concentration,
mp.pollution_severity,
mp.observation_count AS microplastic_sample_count,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_silver.sst_silver sst
LEFT JOIN mcphersonsharyn_silver.cc_silver cc
ON sst.h3_res4 = cc.h3_res4
AND sst.year = cc.year
AND sst.month = cc.month
LEFT JOIN mcphersonsharyn_silver.enso_silver enso
ON sst.year = enso.year
AND sst.month = enso.month
LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
ON sst.h3_res4 = mp.h3_res4
AND sst.month = mp.month;

In [0]:
-- VALIDATION
-- null_sst and null_h3 must be zero
-- null_cc, null_enso, null_microplastics will have some NULLs — expected
SELECT
count(*) AS total_rows,
count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst,
count(*) filter(WHERE chlorophyll_concentration IS NULL) AS null_cc,
count(*) filter(WHERE enso_phase IS NULL) AS null_enso,
count(*) filter(WHERE microplastic_concentration IS NULL) AS null_microplastics,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
count(DISTINCT year) AS years_loaded
FROM mcphersonsharyn_silver.environmental_conditions_silver;